In [3]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.ndimage import gaussian_filter
import pandas as pd
import scipy.optimize
import math
import MDAnalysis as md

def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they 
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values-average)**2, weights=weights)
    return (average, math.sqrt(variance))


path = '/Volumes/Elements/PTM_project/FLNA21/METAD/'

In [4]:
u = md.Universe(path+'processed.pdb',path+'fit1.xtc')

/Users/olivierstreit/miniconda3/lib/python3.7/site-packages/MDAnalysis/topology/guessers.py:80: UserWarning: Failed to guess the mass for the following atom types: 
  warnings.warn("Failed to guess the mass for the following atom types: {}".format(atom_type))
/Users/olivierstreit/miniconda3/lib/python3.7/site-packages/MDAnalysis/topology/PDBParser.py:330: UserWarning: Element information is absent or missing for a few atoms. Elements attributes will not be populated.
  warnings.warn("Element information is absent or missing for a few "


In [11]:
# find frames where SASA closest to 0.65 and cis
sim=1
SASA = np.loadtxt(path+'RMSD_SASA_data/sasaSER2319_{}.xvg'.format(sim),skiprows=25)[:,2]
zeta = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,1]
time = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,0]

t= 1000
t_discard = 200


frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)
cisframes = np.where(np.abs(zeta)<0.52)[0] # 30 deg cutoff

targetSASA = 0.65
diff = np.abs(SASA-targetSASA)
mindiff = np.min(diff)
minidx = np.where(diff==mindiff)[0]

frames = np.intersect1d(minidx,cisframes)

print(time[frames[5]])
print(zeta[frames[5]])
print(SASA[frames[5]])

517555.024583
0.072624
0.65


In [12]:
frames

array([ 30137,  52601,  54643,  65852, 103379, 103511, 167481, 167496,
       173711, 174264, 174693, 174777])

In [13]:
print(frames[5])

103511


In [14]:
frame=103511
u.trajectory[frame]
protein = u.select_atoms('all')
with md.Writer(path+'flna21_sim1_{}ps_highSASA_cis.pdb'.format(int(time[frames[5]])),protein.n_atoms) as W:
    W.write(protein)
print('Done')

Done


In [15]:
# find frames where SASA closest to 0.8 and trans
sim=1
SASA = np.loadtxt(path+'RMSD_SASA_data/sasaSER2319_{}.xvg'.format(sim),skiprows=25)[:,2]
zeta = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,1]
time = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,0]

t= 1000
t_discard = 200


frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)
cisframes = np.where(np.abs(zeta)>2.62)[0] # 150 deg cutoff

targetSASA = 0.8
diff = np.abs(SASA-targetSASA)
mindiff = np.min(diff)
minidx = np.where(diff==mindiff)[0]

frames = np.intersect1d(minidx,cisframes)

print(time[frames[5]])
print(zeta[frames[5]])
print(SASA[frames[5]])

280005.0133
2.625606
0.8


In [16]:
frames

array([ 29120,  37593,  37828,  38511,  46861,  56001,  63526,  66222,
        67324, 140408, 143117, 179283, 181757, 182169, 187262, 188008,
       192376, 192441, 193541, 193817])

In [17]:
frame=56001
u.trajectory[frame]
protein = u.select_atoms('all')
with md.Writer(path+'flnc24_sim1_{}ps_highSASA_trans.pdb'.format(int(time[frames[5]])),protein.n_atoms) as W:
    W.write(protein)
print('Done')

Done
